# M9 · Cold-start, warm-start, transfer & distillation

_AFP-AI · Domain 1 · Ranking & Recommenders_

**Move safely from priors to learned personalization.**

We blend a cold prior with a warm estimate using an evidence-based confidence weight. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
# Setup - CPU-only and deterministic.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(9)

## First, look at cold and warm scores

A cold-start item begins with priors and content features. As evidence count $n$ grows, the blended score moves toward the warm estimate.

In [ ]:
events = pd.DataFrame({"event": ["A", "B", "C"], "cold": [0.010, 0.016, 0.012], "warm": [0.026, 0.020, 0.018], "impressions": [100, 800, 2500], "attends": [1, 14, 55]})

print(events)

## The handoff formula

We use

$$S(n)=(1-w)S_{cold}+wS_{warm},\quad w=\frac{n}{n+k}$$

where $k$ is a pseudo-count that controls how fast trust moves to the warm model.

### Step 1 - Compute confidence weights

Higher impression counts receive larger warm-model weight.

In [ ]:
k = 1000
events["w"] = events["impressions"] / (events["impressions"] + k)

print(events[["event", "impressions", "w"]].round(3))

assert events.loc[2, "w"] > events.loc[0, "w"]

### Step 2 - Blend cold and warm scores

The blended score stays between the cold and warm estimates.

In [ ]:
events["blend"] = (1.0 - events["w"]) * events["cold"] + events["w"] * events["warm"]

print(events[["event", "cold", "warm", "blend"]].round(4))

assert np.all(events["blend"] >= np.minimum(events["cold"], events["warm"]))
assert np.all(events["blend"] <= np.maximum(events["cold"], events["warm"]))

### Step 3 - Apply explicit exit criteria

A campaign exits cold-start only when it has enough exposure and enough outcomes.

In [ ]:
min_impressions = 1000
min_attends = 30
events["warm_ready"] = (events["impressions"] >= min_impressions) & (events["attends"] >= min_attends)

print(events[["event", "impressions", "attends", "warm_ready"]])

assert events.loc[2, "warm_ready"] == True

## Visualize the handoff curve

The confidence weight rises smoothly with evidence instead of flipping abruptly.

In [ ]:
n_grid = np.arange(0, 5001, 100)
w_grid = n_grid / (n_grid + k)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(n_grid, w_grid)
ax.set_xlabel("impressions n")
ax.set_ylabel("warm weight")
ax.set_title("cold-to-warm handoff")
plt.show()

## Practice

Try each in the empty cell below it.

1. Change `k` to 3000 and see how the handoff slows.
2. Add a new event with 0 impressions and verify its blend equals the cold score.
3. Replace the exit criteria with a confidence-weight threshold of 0.7.

In [ ]:
# Your turn:
